# Job Search Platform Efficacy — Modeling

**Target**: `Offer_Received` (binary classification)

**Pipeline**:
1. Upload data & preparation (dummy coding, scaling)
2. sklearn Pipeline for organized modeling
3. Train / Validation / Test splits
4. Fit ensemble models (prediction) + Logistic Regression (causal analysis)
5. Feature importance visualization (causal analysis)
6. Evaluate, compare, and select best model

In [ ]:
# ── Upload your CSV (Google Colab) ──
from google.colab import files
uploaded = files.upload()  # select job_search_cleaned.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
import joblib
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv("job_search_cleaned.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Drop post-offer columns (would leak target info)
leak_cols = ["Time_to_Offer_Days", "Offer_Salary", "Company_Size_Offered",
             "Role_Relevance", "Accepted_Offer"]
df = df.drop(columns=leak_cols)

# Identify column types
target = "Offer_Received"

cat_cols = ["University_Rating", "School_Size", "Region",
            "Major_Category", "Primary_Search_Platform"]
num_cols = [c for c in df.columns if c not in cat_cols and c != target]

print(f"Target: {target}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Numeric features ({len(num_cols)}): {num_cols}")
print(f"\nTarget distribution:\n{df[target].value_counts(normalize=True).round(3)}")

## 2. Bivariate EDA — Exploring Relationships & MLR Assumption Checks

Scatter plots and box plots to reveal:
- **Non-linear relationships** → may need log/poly transforms
- **Heteroscedasticity** → may need interaction terms or robust methods
- **Group differences** across categorical features

In [ ]:
# Numeric bivariate: scatter plots of each numeric feature vs target (jittered)
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.ravel()

for i, col in enumerate(num_cols):
    if i >= len(axes):
        break
    ax = axes[i]
    jitter = np.random.normal(0, 0.05, size=len(df))
    ax.scatter(df[col], df[target] + jitter, alpha=0.05, s=5)
    ax.set_xlabel(col, fontsize=8)
    ax.set_ylabel(target, fontsize=8)
    ax.set_title(f"{col} vs {target}", fontsize=9)

# hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numeric Features vs Offer_Received (jittered)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Bar plots: categorical features vs offer rate
fig, axes = plt.subplots(1, len(cat_cols), figsize=(20, 5))

for i, col in enumerate(cat_cols):
    ct = df.groupby(col)[target].mean().sort_values()
    ct.plot(kind="bar", ax=axes[i], color="steelblue", edgecolor="black")
    axes[i].set_title(f"Offer Rate by {col}", fontsize=9)
    axes[i].set_ylabel("P(Offer)")
    axes[i].tick_params(axis="x", rotation=45, labelsize=7)

plt.suptitle("Categorical Features → Offer Rate", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — check for multicollinearity
corr = df[num_cols + [target]].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Matrix — Numeric Features + Target")
plt.tight_layout()
plt.show()

## 3. Splits & sklearn Pipeline

In [ ]:
# ── X / y split ──
X = df.drop(columns=[target])
y = df[target]

# ── Train (60%) / Validation (20%) / Test (20%) ──
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

print(f"Train:      {X_train.shape[0]:,}  ({y_train.mean():.3f} offer rate)")
print(f"Validation: {X_val.shape[0]:,}  ({y_val.mean():.3f} offer rate)")
print(f"Test:       {X_test.shape[0]:,}  ({y_test.mean():.3f} offer rate)")

In [ ]:
# ── Preprocessor: scale numerics + one-hot encode categoricals ──
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
    ]
)

# ── Build a Pipeline per algorithm ──
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", preprocessor),
        ("clf", GradientBoostingClassifier(n_estimators=200, random_state=42)),
    ]),
    "AdaBoost": Pipeline([
        ("prep", preprocessor),
        ("clf", AdaBoostClassifier(n_estimators=200, random_state=42)),
    ]),
    "Bagging": Pipeline([
        ("prep", preprocessor),
        ("clf", BaggingClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),
}

print(f"Models to fit: {list(models.keys())}")

## 4. Fit All Models & Evaluate on Validation Set

In [ ]:
# ── Fit each model and collect validation metrics ──
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_prob = pipe.predict_proba(X_val)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1": f1_score(y_val, y_pred),
        "AUC": roc_auc_score(y_val, y_prob),
    })
    print(f"✓ {name} fitted")

results_df = pd.DataFrame(results).set_index("Model").sort_values("AUC", ascending=False)
results_df.style.format("{:.4f}").highlight_max(axis=0, color="lightgreen")

## 5. Causal Analysis — Logistic Regression Coefficients

Logistic regression coefficients show the direction and magnitude of each feature's effect on the probability of receiving an offer.

In [ ]:
# ── Extract logistic regression feature names & coefficients ──
lr_pipe = models["Logistic Regression"]
ohe = lr_pipe.named_steps["prep"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + cat_feature_names

coefs = pd.Series(
    lr_pipe.named_steps["clf"].coef_[0],
    index=all_feature_names,
).sort_values()

# ── Horizontal bar chart ──
fig, ax = plt.subplots(figsize=(10, max(6, len(coefs) * 0.3)))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in coefs]
coefs.plot(kind="barh", ax=ax, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Coefficient (standardized)")
ax.set_title("Logistic Regression — Feature Coefficients\n(Causal Direction & Magnitude)", fontsize=13)
ax.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 6. Select Best Model & Final Test Evaluation

In [ ]:
# ── Pick the model with the highest validation AUC ──
best_name = results_df.index[0]
best_pipe = models[best_name]
print(f"Best model (by validation AUC): {best_name}")

# ── Evaluate on held-out TEST set ──
y_test_pred = best_pipe.predict(X_test)
y_test_prob = best_pipe.predict_proba(X_test)[:, 1]

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1": f1_score(y_test, y_test_pred),
    "AUC": roc_auc_score(y_test, y_test_prob),
}

print(f"\n── Test Set Metrics ({best_name}) ──")
for k, v in test_metrics.items():
    print(f"  {k:>10s}: {v:.4f}")

print(f"\n── Classification Report ──")
print(classification_report(y_test, y_test_pred, target_names=["No Offer", "Offer"]))

In [ ]:
# ── Confusion Matrix ──
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred, display_labels=["No Offer", "Offer"],
    cmap="Blues", ax=ax
)
ax.set_title(f"Confusion Matrix — {best_name} (Test Set)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart comparing all models across all metrics ──
fig, ax = plt.subplots(figsize=(12, 6))
results_df.plot(kind="bar", ax=ax, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Validation Set Metrics", fontsize=13)
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save the best model to disk ──
joblib.dump(best_pipe, "best_model.pkl")
print(f"Saved: best_model.pkl  ({best_name})")

# Download the model file (Colab)
from google.colab import files
files.download("best_model.pkl")